# Emotion Recognition — Feature Extraction from saved features
Poetry Vision, part 2 (Renad).

The image features were already extracted in an earlier session (frozen ResNet50 and frozen CLIP, 8000 train / 1600 val / 4000 test images from EmoSet, balanced per emotion). Here I only train the classification head on top of them, which is the feature extraction step from the transfer learning lab. Fine-tuning needs the images, so it's done in `emotion_keras.ipynb`.

No GPU needed: Runtime > Change runtime type > CPU is fine.

In [ ]:
import os, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

CLASSES = ['amusement', 'awe', 'contentment', 'excitement', 'anger', 'disgust', 'fear', 'sadness']
OUT = '/content/emotion_features_results'
os.makedirs(OUT, exist_ok=True)
tf.keras.utils.set_random_seed(42)

## 1) Load the saved features
Upload `emotion_features_backup.zip` (or drag it into the Files panel on the left, then run the cell).

In [ ]:
if not os.path.exists('/content/emotion_features_backup.zip'):
    from google.colab import files
    files.upload()
!unzip -q -o /content/emotion_features_backup.zip -d /content
!ls /content/features

In [ ]:
train = np.load('/content/features/train.npz')
val   = np.load('/content/features/val.npz')
test  = np.load('/content/features/test.npz')
text  = np.load('/content/features/text_embeddings.npz')

y_train, y_val, y_test = train['labels'], val['labels'], test['labels']

print('ResNet50 features:', train['resnet'].shape, val['resnet'].shape, test['resnet'].shape)
print('CLIP features:    ', train['clip'].shape, val['clip'].shape, test['clip'].shape)
print('train per emotion:', dict(zip(CLASSES, np.bincount(y_train, minlength=8))))
print('test per emotion: ', dict(zip(CLASSES, np.bincount(y_test, minlength=8))))

## 2) Before training
- Random head: what the model scores before any training (chance is 1/8 = 0.125).
- CLIP zero-shot: CLIP compares each image with sentences like "a photo that evokes sadness", still without training on EmoSet.

In [ ]:
results = {}

def score(name, probs):
    y_pred = probs.argmax(axis=1)
    acc = float((y_pred == y_test).mean())
    macro_f1 = float(f1_score(y_test, y_pred, average='macro'))
    results[name] = {'test_accuracy': acc, 'test_macro_f1': macro_f1}
    print(f"{name}: Test Accuracy = {acc:.4f} | Macro F1 = {macro_f1:.4f}")
    return y_pred

def build_head(input_dim, x_train):
    norm = tf.keras.layers.Normalization()
    norm.adapt(x_train)                      # scale features using the train set only
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(input_dim,)),
        norm,
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(len(CLASSES), activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# random (untrained) head on ResNet50 features
untrained = build_head(train['resnet'].shape[1], train['resnet'])
score('before_training_random_head', untrained.predict(test['resnet'], verbose=0))

# CLIP zero-shot
logits = float(text['logit_scale']) * test['clip'] @ text['class_text'].T
zs_probs = tf.nn.softmax(logits).numpy()
score('clip_zero_shot', zs_probs)

## 3) Feature extraction: ResNet50 (frozen) + new head

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

resnet_model = build_head(train['resnet'].shape[1], train['resnet'])
resnet_model.summary()

resnet_history = resnet_model.fit(
    train['resnet'], y_train,
    epochs=30, batch_size=64,
    validation_data=(val['resnet'], y_val),
    callbacks=[early_stop]
)
resnet_probs = resnet_model.predict(test['resnet'], verbose=0)
score('resnet50_feature_extraction', resnet_probs)

## 4) Feature extraction: CLIP (frozen) + new head

In [ ]:
clip_model = build_head(train['clip'].shape[1], train['clip'])

clip_history = clip_model.fit(
    train['clip'], y_train,
    epochs=30, batch_size=64,
    validation_data=(val['clip'], y_val),
    callbacks=[early_stop]
)
clip_probs = clip_model.predict(test['clip'], verbose=0)
score('clip_feature_extraction', clip_probs)

## 5) Colour and light only
A head trained only on 5 numbers per image (brightness, contrast, saturation, colourfulness, warmth). Shows how much of the emotion colour and light explain on their own.

In [ ]:
attr_model = build_head(train['attributes'].shape[1], train['attributes'])
attr_model.fit(train['attributes'], y_train, epochs=30, batch_size=64,
               validation_data=(val['attributes'], y_val), callbacks=[early_stop], verbose=0)
score('colour_light_only', attr_model.predict(test['attributes'], verbose=0))

## 6) Results

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(resnet_history.history['accuracy'], label='ResNet50 train')
plt.plot(resnet_history.history['val_accuracy'], label='ResNet50 val')
plt.plot(clip_history.history['accuracy'], label='CLIP train')
plt.plot(clip_history.history['val_accuracy'], label='CLIP val')
plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.subplot(1, 2, 2)
plt.plot(resnet_history.history['val_loss'], label='ResNet50 val loss')
plt.plot(clip_history.history['val_loss'], label='CLIP val loss')
plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.savefig(f'{OUT}/training_curves.png', dpi=150)
plt.show()

In [ ]:
names = list(results.keys())
plt.figure(figsize=(10, 4))
plt.bar(names, [results[n]['test_macro_f1'] for n in names], label='Macro F1')
plt.plot(names, [results[n]['test_accuracy'] for n in names], 'o', color='orange', label='Accuracy')
plt.axhline(1 / len(CLASSES), linestyle=':', color='gray', label='Chance (1/8)')
for i, n in enumerate(names):
    plt.text(i, results[n]['test_macro_f1'] + 0.01, f"{results[n]['test_macro_f1']:.3f}", ha='center')
plt.xticks(rotation=20); plt.ylabel('Test score'); plt.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/model_comparison.png', dpi=150)
plt.show()

In [ ]:
best = max(['resnet50_feature_extraction', 'clip_feature_extraction'], key=lambda n: results[n]['test_macro_f1'])
best_probs = resnet_probs if best == 'resnet50_feature_extraction' else clip_probs
y_pred = best_probs.argmax(axis=1)

report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(best); print(report)
open(f'{OUT}/classification_report.txt', 'w').write(best + '\n' + report)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=CLASSES, normalize='true',
                                        values_format='.2f', cmap='Blues', ax=ax, xticks_rotation=45)
plt.title(f'Confusion Matrix (normalized) — {best}')
plt.tight_layout()
plt.savefig(f'{OUT}/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Colour and light by emotion (test set)
names_attr = ['brightness', 'contrast', 'saturation', 'colourfulness', 'warmth']
plt.figure(figsize=(18, 4))
for k, a in enumerate(names_attr):
    plt.subplot(1, 5, k + 1)
    plt.boxplot([test['attributes'][y_test == c, k] for c in range(len(CLASSES))], showfliers=False)
    plt.xticks(range(1, 9), CLASSES, rotation=60, fontsize=8)
    plt.title(a)
plt.tight_layout()
plt.savefig(f'{OUT}/colour_light_by_emotion.png', dpi=150)
plt.show()

## 7) Save and download

In [ ]:
resnet_model.save(f'{OUT}/resnet50_head.keras')
clip_model.save(f'{OUT}/clip_head.keras')
json.dump(results, open(f'{OUT}/metrics.json', 'w'), indent=2)
print(json.dumps(results, indent=2))

from google.colab import files
!cd /content && rm -f emotion_features_results.zip && zip -qr emotion_features_results.zip emotion_features_results
files.download('/content/emotion_features_results.zip')